In [3]:
import os
import django
import sys
# Set up Django environment
sys.path.append('/nd_deployment/deIdentification')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.models import DbDetailsModel, TableDetailsModel
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process.main import start_de_identification_for_table
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.views.db_views import run_stats_generation_task

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [2]:
TableDetailsModel.objects.filter(table_status=2).last().qc_result

{'table_name': 'zipcodezone',
 'ColumnsQCResult': {},
 'dest_rows_count': 42327,
 'final_qc_result': {'reason': '', 'is_qc_passed': True},
 'ignore_rows_count': 0,
 'source_rows_count': 42327,
 'unstruct_sample_size': 3991}

In [11]:
import os, json
code_failure = []
# table_name, source_row_count, dest_row_count, qc_status, ignore_rows_count, reason, is_phi
all_results = []
def is_phi_table(table_config):
    for col_conf in table_config['columns_details']:
        if col_conf['is_phi']:
            return True
    return False
    
file_path = '/NOTEBOOK/Northwest/AUTO_QC_RESULT'
for table_obj in TableDetailsModel.objects.filter(table_status=2):
    qc_result = table_obj.qc_result
    if not qc_result:
        continue
    one_table = {
        "table_name": table_obj.table_name,
        "source_rows_count": qc_result['source_rows_count'],
        "dest_rows_count": qc_result['dest_rows_count'],
        "qc_status": qc_result.get("final_qc_result", {}).get("is_qc_passed"),
        "ignore_rows_count": qc_result['ignore_rows_count'],
        "reason": qc_result.get("final_qc_result", {}).get("reason"),
        "is_phi": is_phi_table(table_obj.table_details_for_ui)
    }
    all_results.append(one_table)
    # fpath = os.path.join(file_path, table_obj.table_name + ".json")
    # with open(fpath, 'w') as fp:
    #     json.dump(qc_result, fp)


import csv

# Sample list of dictionaries

# Output CSV file path
csv_file = "/NOTEBOOK/Northwest/qc_output_wed_23apr.csv"

# Writing to CSV
with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=all_results[0].keys())
    writer.writeheader()
    writer.writerows(all_results)

In [15]:
import csv

# Sample list of dictionaries

# Output CSV file path
csv_file = "/NOTEBOOK/Northwest/qc_output_wed_23apr.csv"

# Writing to CSV
with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=all_results[0].keys())
    writer.writeheader()
    writer.writerows(all_results)


In [13]:
pwd

'/NOTEBOOK/Northwest'